In [1]:
# pip install hst_funcs
from hst_funcs.finanace.Equity import MC_vanilla

MC_vanilla.MC_Greeks(MC_vanilla.MC_Call, 100, 100, 1, 0.05, 0.3)

{'Price': 14.34948691846546,
 'Delta (Δ)': 0.6268034183046586,
 'Gamma (Γ)': 0.011621810017498686,
 'Theta (Θ)': -0.022432412240196035,
 'Vega (V)': 0.38365709379176793,
 'Rho (ρ)': 0.0048339045589514025}

In [2]:
MC_vanilla.MC_Greeks(MC_vanilla.MC_Put, 100, 100, 1, 0.05, 0.3)

{'Price': 9.275129383003724,
 'Delta (Δ)': -0.37516958155066504,
 'Gamma (Γ)': 0.01162181001753816,
 'Theta (Θ)': -0.009135373917150295,
 'Vega (V)': 0.3772682484415787,
 'Rho (ρ)': -0.004678548225085848}

In [3]:
from hst_funcs.finanace.Equity import BS

BS.bs_call(100, 100, 1, 0.05, 0.3)

14.231254785985819

In [4]:
BS.bs_put(100, 100, 1, 0.05, 0.3)

9.354197236057225

# Explicit FDM

In [5]:
# 유럽형 콜옵션 가격
import numpy as np
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 300
K = 100
T = 1.0
r = 0.05
sigma = 0.3
M = 1000 # 시간 격자
N = 500 # 기초자산 격자

x_max = np.log(S_max)
v = r - 0.5 * sigma**2

dt = T / M
dx = x_max / N
grid = np.zeros((M+1, N+1))
x = np.linspace(0, x_max, N+1)

# 만기 시점 콜옵션 payoff
grid[-1, :] = np.maximum(np.exp(x) - K, 0)

# 계수 계산
pu = dt * ((sigma**2) / (2 * dx**2) + v / (2 * dx))
pm = 1 - dt * (sigma**2 / dx**2) - r * dt
pd = dt * ((sigma**2) / (2 * dx**2) - v / (2 * dx))

# 역방향 계산
for i in reversed(range(M)):
    for j in range(1, N):
        grid[i, j] = pu * grid[i+1, j+1] + pm * grid[i+1, j] + pd * grid[i+1, j-1]
    # 콜옵션의 경계 조건
    grid[i, 0] = 0
    grid[i, N] = np.exp(x_max) - K* np.exp(-r * (T - i*dt))

# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
S = np.exp(x)              # 로그 스케일을 실제 자산 가격으로 복원
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")


유럽형 콜옵션 가격 (S0 = 100): 14.2344


In [6]:
BS.bs_call(100, 100, 1, 0.05, 0.3)

14.231254785985819

In [7]:
# 유럽형 풋옵션 가격
import numpy as np
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 300
K = 100
T = 1.0
r = 0.05
sigma = 0.3
M = 1000 # 시간 격자
N = 500 # 기초자산 격자

x_max = np.log(S_max)
v = r - 0.5 * sigma**2

dt = T / M
dx = x_max / N
grid = np.zeros((M+1, N+1))
x = np.linspace(0, x_max, N+1)

# 만기 시점 풋옵션 payoff
grid[-1, :] = np.maximum(K-np.exp(x), 0)

# 계수 계산
pu = dt * ((sigma**2) / (2 * dx**2) + v / (2 * dx))
pm = 1 - dt * (sigma**2 / dx**2) - r * dt
pd = dt * ((sigma**2) / (2 * dx**2) - v / (2 * dx))

# 역방향 계산
for i in reversed(range(M)):
    for j in range(1, N):
        grid[i, j] = pu * grid[i+1, j+1] + pm * grid[i+1, j] + pd * grid[i+1, j-1]
    # 풋옵션 경계 조건
    grid[i, N] = 0
    grid[i, 0] = K* np.exp(-r * (T - i*dt))

# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
S = np.exp(x)              # 로그 스케일을 실제 자산 가격으로 복원
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 풋옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 풋옵션 가격 (S0 = 100): 9.3572


In [25]:
BS.bs_put(100, 100, 1, 0.05, 0.3)

9.354197236057225

## 공간격자 변환시의 Explicit FDM

In [21]:
# 유럽형 콜옵션 가격
import numpy as np
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 300
K = 100
T = 1.0
r = 0.05
q = 0
sigma = 0.3
M = 1000 # 시간 격자
N = 100 # 기초자산 격자

dt = T / M
dS = S_max / N
grid = np.zeros((M+1, N+1))
S = np.linspace(0, S_max, N+1)

# 만기 시점 콜옵션 payoff
grid[-1, :] = np.maximum(S - K, 0)


# 역방향 계산
for i in reversed(range(M)):
    for j in range(1, N):
        # Coefficients
        Uj = dt * (0.5 * sigma**2 * j**2 + 0.5 * (r - q) * j)
        Mj = 1 - r * dt - dt * sigma**2 * j**2
        Dj = dt * (0.5 * sigma**2 * j**2 - 0.5 * (r - q) * j)
        grid[i, j] = Uj * grid[i+1, j+1] + Mj * grid[i+1, j] + Dj* grid[i+1, j-1]
    # 콜옵션의 경계 조건
    grid[i, 0] = 0
    grid[i, N] = S_max - K* np.exp(-r * (T - i*dt))

# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 콜옵션 가격 (S0 = 100): 14.2437


In [22]:
# 유럽형 풋옵션 가격

dt = T / M
dS = S_max / N
grid = np.zeros((M+1, N+1))
S = np.linspace(0, S_max, N+1)

# 만기 시점 풋옵션 payoff
grid[-1, :] = np.maximum(K - S, 0)


# 역방향 계산
for i in reversed(range(M)):
    for j in range(1, N):
        # Coefficients
        Uj = dt * (0.5 * sigma**2 * j**2 + 0.5 * (r - q) * j)
        Mj = 1 - r * dt - dt * sigma**2 * j**2
        Dj = dt * (0.5 * sigma**2 * j**2 - 0.5 * (r - q) * j)
        grid[i, j] = Uj * grid[i+1, j+1] + Mj * grid[i+1, j] + Dj* grid[i+1, j-1]
    # 풋옵션의 경계 조건
    grid[i, 0] = K* np.exp(-r * (T - i*dt))
    grid[i, N] = 0

# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 풋옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 풋옵션 가격 (S0 = 100): 9.3666


# Implicit FDM

In [9]:
# 삼중대각행렬 예제
from scipy.linalg import solve_banded
import numpy as np

# 예: 3x3 삼중대각 행렬
# A = [[2, 1, 0],
#      [1, 2, 1],
#      [0, 1, 2]]
a = np.array([1, 1])   # 하단 대각
b = np.array([2, 2, 2]) # 주 대각
c = np.array([1, 1])   # 상단 대각

# ab 구성
ab = np.zeros((3, 3))
ab[0, 1:] = c
ab[1, :] = b
ab[2, :-1] = a

# Ax = d
d = np.array([4, 5, 6])

x = solve_banded((1, 1), ab, d)
print(x)


[2.00000000e+00 2.96059473e-16 3.00000000e+00]


In [10]:
from scipy.linalg import solve_banded
from sympy import Matrix 

# Ax = d
d = np.array([4, 5, 6])

x = solve_banded((1, 1), ab, d)
Matrix(x)

Matrix([
[                 2.0],
[2.96059473233375e-16],
[                 3.0]])

In [35]:
# 유럽형 콜옵션 평가

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

# 파라미터 설정
S_max = 400
M = 2000        # 시간 스텝 수
N =2000      # 공간 스텝 수 (로그 공간)
K = 100
T = 1.0
r = 0.05
sigma = 0.3


# 로그 공간 설정
x_max = np.log(S_max)
dx = x_max / N
dt = T / M
x = np.linspace(0, x_max, N+1)
v = r - 0.5 * sigma**2

# payoff (만기 시점)
grid = np.zeros((M+1, N+1))
grid[-1, :] = np.maximum(np.exp(x) - K, 0)

# 계수 계산 (전 구간에서 상수로 적용)
pu = -dt * (sigma**2 / (2 * dx**2) + v / (2 * dx))
pm = 1 + dt * (sigma**2 / (dx**2) + r)
pd = -dt * (sigma**2 / (2 * dx**2) - v / (2 * dx))

# 삼중대각 행렬 구성 (banded 형식)
ab = np.zeros((3, N-1))
ab[0, 1:] = pu     # 위쪽 대각
ab[1, :]  = pm     # 메인 대각
ab[2, :-1] = pd    # 아래쪽 대각

# 시간 역순으로 계산
for i in reversed(range(M)):
    rhs = grid[i+1, 1:N]  # 다음 시점의 내부 값
    # 경계 조건 보정
    rhs[0]   -= pd * 0                             # S=0 → C=0
    rhs[-1]  -= pu * (np.exp(x_max) - K*np.exp(-r*dt*(M-i)))  # S=S_max → C=S−K e^(-rτ)
    grid[i, 1:N] = solve_banded((1, 1), ab, rhs)

# 초기 자산 가격에서 보간
S0 = 100
S = np.exp(x)
C0 = grid[0, :]
option_price = np.interp(S0, S, C0)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 콜옵션 가격 (S0 = 100): 14.2305


In [21]:
BS.bs_call(100, 100, 1, 0.05, 0.3)

14.231254785985819

In [32]:
# 만기 직전 시점(M-1)에서의 격자해 구하기
rhs = grid[M, 1:N]  
# 경계 조건 보정
rhs[0]   -= pd * 0                             # S=0 → C=0
rhs[-1]  -= pu * (np.exp(x_max) - K*np.exp(-r*dt))  # S=S_max → C=S−K e^(-rτ)
grid[M-1, 1:N] = solve_banded((1, 1), ab, rhs)

In [34]:
# M-2 시점에서의 격자해 구하기
rhs = grid[M-1, 1:N]  
# 경계 조건 보정
rhs[0]   -= pd * 0                             # S=0 → C=0
rhs[-1]  -= pu * (np.exp(x_max) - K*np.exp(-r*dt))  # S=S_max → C=S−K e^(-rτ)
grid[M-2, 1:N] = solve_banded((1, 1), ab, rhs)

In [36]:
# 유럽형 풋옵션
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

# 파라미터 설정
S_max = 400
K = 100
T = 1.0
r = 0.05
sigma = 0.3
M = 2000         # 시간 스텝 수
N = 2000          # 공간 스텝 수 (로그 공간)

# 로그 공간 설정
x_max = np.log(S_max)
dx = x_max / N
dt = T / M
x = np.linspace(0, x_max, N+1)
v = r - 0.5 * sigma**2

# payoff (만기 시점)
grid = np.zeros((M+1, N+1))
grid[-1, :] = np.maximum(K-np.exp(x), 0)

# 계수 계산 (전 구간에서 상수로 적용)
pu = -dt * (sigma**2 / (2 * dx**2) + v / (2 * dx))
pm = 1 + dt * (sigma**2 / (dx**2) + r)
pd = -dt * (sigma**2 / (2 * dx**2) - v / (2 * dx))

# 삼중대각 행렬 구성 (banded 형식)
ab = np.zeros((3, N-1))
ab[0, 1:] = pu     # 위쪽 대각
ab[1, :]  = pm     # 메인 대각
ab[2, :-1] = pd    # 아래쪽 대각

# 시간 역순으로 계산
for i in reversed(range(M)):
    rhs = grid[i+1, 1:N]  # 다음 시점의 내부 값
    # 경계 조건 보정
    rhs[0]   -= pd *  (K*np.exp(-r*dt*(M-i)))  #  C=K e^(-rτ)                         
    rhs[-1]  -= pu * 0   # S=Smax → C=0
    grid[i, 1:N] = solve_banded((1, 1), ab, rhs)

# 초기 자산 가격에서 보간
S0 = 100
S = np.exp(x)
C0 = grid[0, :]
option_price = np.interp(S0, S, C0)

print(f"유럽형 풋옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 풋옵션 가격 (S0 = 100): 9.3535


In [37]:
BS.bs_put(100, 100, 1, 0.05, 0.3)

9.354197236057225

## 공간격자 변환시의 Implicit FDM

In [39]:
# 유럽형 콜옵션 가격
import numpy as np
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 300
K = 100
T = 1.0
r = 0.05
q = 0
sigma = 0.3
M = 2000 # 시간 격자
N = 2000 # 기초자산 격자

dt = T / M
dS = S_max / N
grid = np.zeros((M+1, N+1))
S = np.linspace(0, S_max, N+1)

# 만기 시점 콜옵션 payoff
grid[-1, :] = np.maximum(S - K, 0)

# Coefficients from the derived implicit FDM scheme
j=np.arange(1,N)
Uj= dt * (-0.5 * sigma**2 * j**2 - 0.5 * (r - q) * j)
Mj = 1 + dt * (r + sigma**2 * j**2)
Dj = dt * (-0.5 * sigma**2 * j**2 + 0.5 * (r - q) * j)

# 삼중대각 행렬 구성 (banded 형식)
ab = np.zeros((3, N-1))
ab[0, 1:] = Uj[:-1]     # 위쪽 대각
ab[1, :]  = Mj     # 메인 대각
ab[2, :-1] = Dj[1:]    # 아래쪽 대각

# 역방향 계산
for i in reversed(range(M)):
    rhs = grid[i+1, 1:N]  # 다음 시점의 내부 값

    rhs[0]   -= Dj[0] * 0                             # S=0 → C=0
    rhs[-1]  -= Uj[-1] * (S_max - K*np.exp(-r*dt*(M-i)))  # S=S_max → C=S−K e^(-rτ)
    grid[i, 1:N] = solve_banded((1, 1), ab, rhs)

# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 콜옵션 가격 (S0 = 100): 14.2305


In [3]:
# 유럽형 콜옵션 가격 (경계조건에서의 감마이용)
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

# 파라미터 설정
S_max = 500
K = 100
T = 1.0
r = 0.05
q = 0
sigma = 0.3
M = 2000 # 시간 격자
N = 2000 # 기초자산 격자

dt = T / M
dS = S_max / N
grid = np.zeros((M+1, N+1))
S = np.linspace(0, S_max, N+1)

# 만기 시점 콜옵션 payoff
grid[-1, :] = np.maximum(S - K, 0)

# Coefficients from the derived implicit FDM scheme
j=np.arange(1,N)
Uj= dt * (-0.5 * sigma**2 * j**2 - 0.5 * (r - q) * j)
Mj = 1 + dt * (r + sigma**2 * j**2)
Dj = dt * (-0.5 * sigma**2 * j**2 + 0.5 * (r - q) * j)

# 삼중대각 행렬 구성 (banded 형식)
ab = np.zeros((3, N-1))
ab[0, 1:] = Uj[:-1]     # 위쪽 대각
ab[1, :]  = Mj     # 메인 대각
ab[2, :-1] = Dj[1:]    # 아래쪽 대각

# 역방향 계산
for i in reversed(range(M)):
    rhs = grid[i+1, 1:N]  # 다음 시점의 내부 값

    rhs[0]   -= Dj[0] * (2*grid[i, 1] - grid[i, 2] )
    rhs[-1]  -= Uj[-1] * (2*grid[i, -2] - grid[i, -3] )
    grid[i, 1:N] = solve_banded((1, 1), ab, rhs)
    
# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 콜옵션 가격 (S0 = 100): 14.2303


In [4]:
# 유럽형 풋옵션 가격
import numpy as np
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 350
K = 100
T = 1.0
r = 0.05
q = 0
sigma = 0.3
M = 2000 # 시간 격자
N = 2000 # 기초자산 격자

dt = T / M
dS = S_max / N
grid = np.zeros((M+1, N+1))
S = np.linspace(0, S_max, N+1)

# 만기 시점 풋옵션 payoff
grid[-1, :] = np.maximum(K - S, 0)

# Coefficients from the derived implicit FDM scheme
j=np.arange(1,N)
Uj= dt * (-0.5 * sigma**2 * j**2 - 0.5 * (r - q) * j)
Mj = 1 + dt * (r + sigma**2 * j**2)
Dj = dt * (-0.5 * sigma**2 * j**2 + 0.5 * (r - q) * j)

# 삼중대각 행렬 구성 (banded 형식)
ab = np.zeros((3, N-1))
ab[0, 1:] = Uj[:-1]     # 위쪽 대각
ab[1, :]  = Mj     # 메인 대각
ab[2, :-1] = Dj[1:]    # 아래쪽 대각

# 역방향 계산
for i in reversed(range(M)):
    rhs = grid[i+1, 1:N]  # 다음 시점의 내부 값

    rhs[0]   -= Dj[0] * (2*grid[i, 1] - grid[i, 2] )
    rhs[-1]  -= Uj[-1] * (2*grid[i, -2] - grid[i, -3] )
    grid[i, 1:N] = solve_banded((1, 1), ab, rhs)
    
# 초기 자산 가격에서 옵션 가격 추출
S0 = 100
C0_line = grid[0, :]       # t = 0에서의 옵션 가격 분포
option_price = np.interp(S0, S, C0_line)

print(f"유럽형 콜옵션 가격 (S0 = {S0}): {option_price:.4f}")

유럽형 콜옵션 가격 (S0 = 100): 9.3535


# Crank-Nicholson FDM

In [46]:
# 유럽형 콜옵션
import numpy as np
from scipy.linalg import solve_banded
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 400
M = 1000     # 시간 스텝 수
N = 1000     # 공간 스텝 수 (로그 공간)
K = 100
T = 1.0
r = 0.05
sigma = 0.3


# 로그 공간 기준 설정
x_max = np.log(S_max)
dx = x_max / N
dt = T / M
x = np.linspace(0, x_max, N+1)
v = r - 0.5 * sigma**2

# payoff (만기 시점)
grid = np.zeros((M+1, N+1))
S = np.exp(x)
grid[-1, :] = np.maximum(S - K, 0)

# Crank-Nicolson 계수 정의
pu = -dt/4 * ((sigma**2 / dx**2) + v / dx)
pm = 1 + dt/2 * (sigma**2 / dx**2 + r)
pd = -dt/4 * ((sigma**2 / dx**2) - v / dx)

# Banded 행렬 A (i 시점 계수)
ab_A = np.zeros((3, N-1))
ab_A[0, 1:] = pu         # 상단 대각
ab_A[1, :]  = pm         # 주대각
ab_A[2, :-1] = pd        # 하단 대각

# B 행렬 계수 (i+1 시점 우변 계수)
pu_B = -pu
pm_B = 2-pm
pd_B = -pd

# 시간 역방향 계산
for i in reversed(range(M)):
    rhs = np.zeros(N-1)
    
    #오른쪽 행렬 곱 계산
    for j in range(1, N):
        rhs[j-1] = (
            pu_B * grid[i+1, j+1] +
            pm_B * grid[i+1, j] +
            pd_B * grid[i+1, j-1]
        )
    # 경계 조건 보정
    rhs[0]  -= pd * 0
    C_R = np.exp(x_max) - K * np.exp(-r * (T - i * dt))
    rhs[-1] -= pu * C_R

    # 현재 시점 옵션 가격 계산
    grid[i, 1:N] = solve_banded((1, 1), ab_A, rhs)

# 초기 자산 가격에서 보간하여 옵션 가격 추출
S0 = 100
C0 = grid[0, :]
option_price = np.interp(S0, S, C0)
option_price

14.231195295366062

In [40]:
BS.bs_call(100, 100, 1, 0.05, 0.3)

14.231254785985819

In [47]:
# 유럽형 풋옵션
import numpy as np
from scipy.linalg import solve_banded
import matplotlib.pyplot as plt

# 파라미터 설정
S_max = 400
M = 1000     # 시간 스텝 수
N = 1000     # 공간 스텝 수 (로그 공간)
K = 100
T = 1.0
r = 0.05
sigma = 0.3


# 로그 공간 기준 설정
x_max = np.log(S_max)
dx = x_max / N
dt = T / M
x = np.linspace(0, x_max, N+1)
v = r - 0.5 * sigma**2

# payoff (만기 시점)
grid = np.zeros((M+1, N+1))
S = np.exp(x)
grid[-1, :] = np.maximum(K-S, 0)

# Crank-Nicolson 계수 정의
pu = -dt/4 * ((sigma**2 / dx**2) + v / dx)
pm = 1 + dt/2 * (sigma**2 / dx**2 + r)
pd = -dt/4 * ((sigma**2 / dx**2) - v / dx)

# Banded 행렬 A (i 시점 계수)
ab_A = np.zeros((3, N-1))
ab_A[0, 1:] = pu         # 상단 대각
ab_A[1, :]  = pm         # 주대각
ab_A[2, :-1] = pd        # 하단 대각

# B 행렬 계수 (i+1 시점 우변 계수)
pu_B = -pu
pm_B = 2-pm
pd_B = -pd

# 시간 역방향 계산
for i in reversed(range(M)):
    rhs = np.zeros(N-1)
    
    #오른쪽 행렬 곱 계산
    for j in range(1, N):
        rhs[j-1] = (
            pu_B * grid[i+1, j+1] +
            pm_B * grid[i+1, j] +
            pd_B * grid[i+1, j-1]
        )
    # 경계 조건 보정
    C_R = K * np.exp(-r * (T - i * dt))
    rhs[0]  -= pd * C_R
    rhs[-1] -= pu * 0

    # 현재 시점 옵션 가격 계산
    grid[i, 1:N] = solve_banded((1, 1), ab_A, rhs)

# 초기 자산 가격에서 보간하여 옵션 가격 추출
S0 = 100
C0 = grid[0, :]
option_price = np.interp(S0, S, C0)
option_price

9.354711339758106

In [48]:
BS.bs_put(100, 100, 1, 0.05, 0.3)

9.354197236057225

# FDM Greeks

In [96]:
def FDM_CN_Vanilla_Call(S,K,T,r,sigma,M=1000,N=1000):
    import numpy as np
    from scipy.linalg import solve_banded

    # 기초자산 최대값 
    S_max =S*5

    # 로그 공간 기준 설정
    x_max = np.log(S_max)
    dx = x_max / N
    dt = T / M
    x = np.linspace(0, x_max, N+1)
    v = r - 0.5 * sigma**2

    # payoff (만기 시점)
    grid = np.zeros((M+1, N+1))
    S = np.exp(x)
    grid[-1, :] = np.maximum(S - K, 0)

    # Crank-Nicolson 계수 정의
    pu = -dt/4 * ((sigma**2 / dx**2) + v / dx)
    pm = 1 + dt/2 * (sigma**2 / dx**2 + r)
    pd = -dt/4 * ((sigma**2 / dx**2) - v / dx)

    # Banded 행렬 A (i 시점 계수)
    ab_A = np.zeros((3, N-1))
    ab_A[0, 1:] = pu         # 상단 대각
    ab_A[1, :]  = pm         # 주대각
    ab_A[2, :-1] = pd        # 하단 대각

    # B 행렬 계수 (i+1 시점 우변 계수)
    pu_B = -pu
    pm_B = 2-pm
    pd_B = -pd

    # 시간 역방향 계산
    for i in reversed(range(M)):
        rhs = np.zeros(N-1)

        #오른쪽 행렬 곱 계산
        for j in range(1, N):
            rhs[j-1] = (
                pu_B * grid[i+1, j+1] +
                pm_B * grid[i+1, j] +
                pd_B * grid[i+1, j-1]
            )
        # 경계 조건 보정
        rhs[0]  -= pd * 0
        C_R = np.exp(x_max) - K * np.exp(-r * (T - i * dt))
        rhs[-1] -= pu * C_R

        # 현재 시점 옵션 가격 계산
        grid[i, 1:N] = solve_banded((1, 1), ab_A, rhs)

    # 초기 자산 가격에서 보간하여 옵션 가격 추출
    S0 = 100
    C0 = grid[0, :]
    option_price = np.interp(S0, S, C0)
    
    from scipy.interpolate import CubicSpline

    spline = CubicSpline(S, C0)

    # 1차 및 2차 도함수
    delta = spline(S0, 1)  # 1차 도함수 → Delta
    gamma = spline(S0, 2)  # 2차 도함수 → Gamma
    
    idx = np.searchsorted(S, S0)
    theta = (grid[1, idx] - C0[idx]) / dt
    theta=theta/365 # 1day theta
    
    return option_price,delta.item(),gamma.item(),theta

In [97]:
option_price,delta,gamma,theta=FDM_CN_Vanilla_Call(100,100,1,0.05,0.3)
option_price,delta,gamma,theta

(14.230771608190398,
 0.6242470023134887,
 0.012648476782560087,
 -0.022296318805819376)

In [16]:
def FDM_Greeks(fun,S,K,T,r,sigma,M=1000,N=1000):
    C_0, Delta, Gamma, Theta = fun(S, K, T, r, sigma, M, N)

    # Vega 계산
    eps = sigma * 0.01
    C_sigma_plus, _, _, _ = fun(S, K, T, r, sigma + eps, M, N)
    C_sigma_minus, _, _, _ = fun(S, K, T, r, sigma - eps, M, N)
    Vega = (C_sigma_plus - C_sigma_minus) / (2 * eps)
    Vega=Vega/100 # 1% 베가

    # Rho 계산
    C_r_plus, _, _, _ = fun(S, K, T, r + 0.01, sigma, M, N)
    C_r_minus, _, _, _ = fun(S, K, T, r - 0.01, sigma, M, N)
    Rho = (C_r_plus - C_r_minus) / (2 * 0.01)
    Rho=Rho/10000 # 1bp 르호

    return {
        "Price": C_0,
        "Delta (Δ)": Delta,
        "Gamma (Γ)": Gamma,
        "Theta (Θ)": Theta,
        "Vega (V)": Vega,
        "Rho (ρ)": Rho,
    }

In [102]:
FDM_Greeks(FDM_CN_Vanilla_Call,100,100,1,0.05,0.3)

{'Price': 14.230771608190398,
 'Delta (Δ)': 0.6242470023134887,
 'Gamma (Γ)': 0.012648476782560087,
 'Theta (Θ)': -0.022296318805819376,
 'Vega (V)': 0.379431244507451,
 'Rho (ρ)': 0.004819157262797873}

In [105]:
print('콜옵션의 현재가:',BS.bs_call(100,100,1,0.05,0.3))
print('콜옵션의 델타:',BS.call_delta(100,100,1,0.05,0.3))
print('콜옵션의 감마:',BS.call_gamma(100,100,1,0.05,0.3))
print('콜옵션의 1day 세타:',BS.call_theta(100,100,1,0.05,0.3))
print('콜옵션의 1% 베가:',BS.call_vega(100,100,1,0.05,0.3))
print('콜옵션의 1bp 르호:',BS.call_rho(100,100,1,0.05,0.3))

콜옵션의 현재가: 14.231254785985819
콜옵션의 델타: 0.6242517279060125
콜옵션의 감마: 0.012647764437231512
콜옵션의 1day 세타: -0.0221950408136574
콜옵션의 1% 베가: 0.3794329331169454
콜옵션의 1bp 르호: 0.004819391800461543


In [106]:
def FDM_CN_Vanilla_Put(S,K,T,r,sigma,M=1000,N=1000):
    import numpy as np
    from scipy.linalg import solve_banded

    # 기초자산 최대값 
    S_max =S*5

    # 로그 공간 기준 설정
    x_max = np.log(S_max)
    dx = x_max / N
    dt = T / M
    x = np.linspace(0, x_max, N+1)
    v = r - 0.5 * sigma**2

    # payoff (만기 시점)
    grid = np.zeros((M+1, N+1))
    S = np.exp(x)
    grid[-1, :] = np.maximum(K-S, 0)

    # Crank-Nicolson 계수 정의
    pu = -dt/4 * ((sigma**2 / dx**2) + v / dx)
    pm = 1 + dt/2 * (sigma**2 / dx**2 + r)
    pd = -dt/4 * ((sigma**2 / dx**2) - v / dx)

    # Banded 행렬 A (i 시점 계수)
    ab_A = np.zeros((3, N-1))
    ab_A[0, 1:] = pu         # 상단 대각
    ab_A[1, :]  = pm         # 주대각
    ab_A[2, :-1] = pd        # 하단 대각

    # B 행렬 계수 (i+1 시점 우변 계수)
    pu_B = -pu
    pm_B = 2-pm
    pd_B = -pd

    # 시간 역방향 계산
    for i in reversed(range(M)):
        rhs = np.zeros(N-1)

        #오른쪽 행렬 곱 계산
        for j in range(1, N):
            rhs[j-1] = (
                pu_B * grid[i+1, j+1] +
                pm_B * grid[i+1, j] +
                pd_B * grid[i+1, j-1]
            )
        # 경계 조건 보정
        C_R = K * np.exp(-r * (T - i * dt))
        rhs[0]  -= pd * C_R
        rhs[-1] -= pu * 0

        # 현재 시점 옵션 가격 계산
        grid[i, 1:N] = solve_banded((1, 1), ab_A, rhs)

    # 초기 자산 가격에서 보간하여 옵션 가격 추출
    S0 = 100
    C0 = grid[0, :]
    option_price = np.interp(S0, S, C0)
    
    from scipy.interpolate import CubicSpline

    spline = CubicSpline(S, C0)

    # 1차 및 2차 도함수
    delta = spline(S0, 1)  # 1차 도함수 → Delta
    gamma = spline(S0, 2)  # 2차 도함수 → Gamma
    
    idx = np.searchsorted(S, S0)
    theta = (grid[1, idx] - C0[idx]) / dt
    theta=theta/365 # 1day theta
    
    return option_price,delta.item(),gamma.item(),theta

In [107]:
FDM_Greeks(FDM_CN_Vanilla_Put,100,100,1,0.05,0.3)

{'Price': 9.353713281027286,
 'Delta (Δ)': -0.3757500616135875,
 'Gamma (Γ)': 0.012648999891117058,
 'Theta (Θ)': -0.009266160920153014,
 'Vega (V)': 0.3794480838355815,
 'Rho (ρ)': -0.004693331161065144}

In [108]:
print('풋옵션의 현재가:',BS.bs_put(100,100,1,0.05,0.3))
print('풋옵션의 델타:',BS.put_delta(100,100,1,0.05,0.3))
print('풋옵션의 감마:',BS.put_gamma(100,100,1,0.05,0.3))
print('풋옵션의 1day 세타:',BS.put_theta(100,100,1,0.05,0.3))
print('풋옵션의 1% 베가:',BS.put_vega(100,100,1,0.05,0.3))
print('풋옵션의 1bp 르호:',BS.put_rho(100,100,1,0.05,0.3))

풋옵션의 현재가: 9.354197236057225
풋옵션의 델타: -0.37574827209398753
풋옵션의 감마: 0.012647764437231512
풋옵션의 1day 세타: -0.009164500752003785
풋옵션의 1% 베가: 0.3794329331169454
풋옵션의 1bp 르호: -0.004692902444545599
